In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision import transforms

from NST_Code.utils.models import VGGEncoder

In [ ]:
ROOT = Path.cwd() / "NST_Code"

EXAMPLES = ROOT / "examples"

VGG_WEIGHTS = ROOT / "vgg_normalised.pth"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

image_size = 512

print("Project directory:", ROOT)
print("VGG weights:", VGG_WEIGHTS)
print("Using device:", device)

SyntaxError: unterminated string literal (detected at line 17) (4038696307.py, line 17)

In [ ]:
image_paths = {
    "Original Brad Pitt": EXAMPLES / "brad_pitt.jpg",
    "Stylized Brad Pitt A": EXAMPLES / "stylized_brad_pitt.jpg",
    "Stylized Brad Pitt B": EXAMPLES / "stylized_brad_pitt (1).jpg",
}

for name, path in image_paths.items():
    print(f"{name}: {path.exists()}")

assert VGG_WEIGHTS.exists(), "VGG weights file not found!"

for name, path in image_paths.items():
    assert path.exists(), f"Image not found: {path}"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

In [ ]:
encoder = VGGEncoder(str(VGG_WEIGHTS))

encoder = encoder.to(device)

encoder.eval()

print("VGG encoder loaded successfully!")

In [ ]:
def load_image(path):
    image = Image.open(path).convert("RGB")

    tensor = transform(image)

    tensor = tensor.unsqueeze(0)

    tensor = tensor.to(device)

    return image, tensor

In [ ]:
def show_images(paths_dict):
    fig, axes = plt.subplots(
        1,
        len(paths_dict),
        figsize=(16, 5)
    )

    for ax, (title, path) in zip(axes, paths_dict.items()):
        image = Image.open(path).convert("RGB")

        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_images(image_paths)

In [ ]:
def extract_features(tensor):
    with torch.no_grad():
        h1, h2, h3, h4 = encoder(tensor)

    return {
        "relu1_1": h1,
        "relu2_1": h2,
        "relu3_1": h3,
        "relu4_1": h4,
    }

In [ ]:
image_tensors = {}

feature_bank = {}

for name, path in image_paths.items():
    _, tensor = load_image(path)

    image_tensors[name] = tensor

    feature_bank[name] = extract_features(tensor)

print("Feature extraction completed!")

In [ ]:
for name, features in feature_bank.items():
    print(name)

    for layer, feature in features.items():
        print(f"  {layer}: {tuple(feature.shape)}")

    print()

In [ ]:
def activation_map(feature_tensor):
    activation = feature_tensor[0].mean(dim=0).detach().cpu()

    activation = activation - activation.min()

    activation = activation / (activation.max() + 1e-8)

    return activation

In [ ]:
layers_to_show = [
    "relu1_1",
    "relu2_1",
    "relu3_1",
    "relu4_1",
]

row_labels = ["Input Image"] + layers_to_show

num_rows = len(row_labels)
num_cols = len(image_paths)

fig, axes = plt.subplots(
    num_rows,
    num_cols,
    figsize=(5 * num_cols, 3.6 * num_rows),
    squeeze=False,
)

# Display the original and stylized images
for col, (name, path) in enumerate(image_paths.items()):
    image = Image.open(path).convert("RGB")

    axes[0, col].imshow(image)
    axes[0, col].set_title(name, fontsize=12)
    axes[0, col].axis("off")

# Display activation maps for each image
for row, layer in enumerate(layers_to_show, start=1):
    for col, name in enumerate(image_paths):
        activation = activation_map(feature_bank[name][layer])

        axes[row, col].imshow(activation, cmap="viridis")
        axes[row, col].axis("off")

# Label each row
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(
        label,
        fontsize=11,
        rotation=90,
        labelpad=12,
    )

plt.suptitle(
    "VGG Feature Activations: Original vs. Stylized Images",
    fontsize=15,
    y=1.02,
)

plt.tight_layout()
plt.show()

In [ ]:
print("Number of input images:", len(image_tensors))
print("Number of feature sets:", len(feature_bank))

for name, features in feature_bank.items():
    print(f"{name}: {len(features)} feature maps")

print("\nExperiment completed successfully!")